# Step 11 – Comparison with Hansen Global Forest Change: agreement by tree-cover threshold

Paper 2: *Benchmarking Ten Machine-Learning Algorithms for Mediterranean Forest Mapping with Sentinel-2* (Remote Sensing, 2026).

- **In the paper:** Sections 2.7, 3.7 · Table 11, Figure 14
- **Needs:** CatBoost map tiles (`DATA_DIR/maps/CatBoost/`) and `DATA_DIR/P2_hansen_10m.tif`
- **Writes:** `RESULTS_DIR/hansen_comparison/`

Run the configuration cell below first.

In [ ]:
# ---------------------------------------------------------------- paths
# The data are not included in this repository. Set the two folders below to
# where the files are on your computer (or on your Google Drive in Colab).
#   DATA_DIR    : input data (training samples, Sentinel-2 stack, map tiles, rasters)
#   RESULTS_DIR : outputs written by the notebooks. Later steps read the outputs
#                 of earlier ones (e.g. RESULTS_DIR/train_test_gap/ from step 04).
import os
DATA_DIR    = 'data/'
RESULTS_DIR = 'results/'

# Google Colab: mount Drive and point the folders to it, for example
# from google.colab import drive; drive.mount('/content/drive')
# DATA_DIR    = '/content/drive/MyDrive/<your_folder>/'
# RESULTS_DIR = '/content/drive/MyDrive/<your_folder>/results/'

os.makedirs(RESULTS_DIR, exist_ok=True)

# Files and folders used by this notebook:
#   DATA_DIR / P2_hansen_10m.tif
#   DATA_DIR / maps/CatBoost/
#   RESULTS_DIR / hansen_comparison/

In [ ]:
"""
CatBoost map vs Hansen Global Forest Change

Run after the CatBoost map (DATA_DIR/maps/CatBoost/) and P2_hansen_10m.tif exist.
Can run in the same session as steps 12 and 13.   In Colab first:  !pip install -q rasterio

One pass over the map tiles computes, pixel by pixel on the common 10 m grid:
  (1)  agreement between CatBoost forest and Hansen forest at tree-cover thresholds
         10, 20, 30, 50 % (loss 2001-2024 removed, as in the main comparison)
  (2)  Hansen gain (2000-2012) and loss (2001-2024) pixels: how much of them CatBoost
         maps as forest, and how much of the CatBoost-only area they explain
  (3)  agreement areas (km2) at the main threshold, for Table 5
Areas = number of 10 m pixels x 100 m2.

Outputs (OUT_DIR): hansen_threshold_agreement.csv, hansen_gain_loss.csv,
                   Fig_hansen_thresholds.(png|pdf)
"""


import os, glob
import numpy as np
import pandas as pd
import rasterio
import matplotlib.pyplot as plt
from rasterio.windows import from_bounds

# ------------------------------------------------------------------ config
MAP_DIR     = DATA_DIR + 'maps/CatBoost/'
HANSEN_PATH = DATA_DIR + 'P2_hansen_10m.tif'
OUT_DIR     = RESULTS_DIR + 'hansen_comparison/'
FOREST_CODE = 1
NODATA      = 255
THRESHOLDS  = [10, 20, 30, 50]
MAIN_THRESHOLD = 30          # threshold of the main comparison (Table 5)
LOSS_UNTIL  = 24             # loss 2001..2024 removed from Hansen forest (0 = keep)

os.makedirs(OUT_DIR, exist_ok=True)
tiles = sorted(glob.glob(MAP_DIR + '*.tif'))
assert tiles, f'No CatBoost map tiles in {MAP_DIR}'
hansen = rasterio.open(HANSEN_PATH)

acc = {t: dict.fromkeys(['both', 'cb_only', 'h_only', 'neither'], 0) for t in THRESHOLDS}
gl = dict.fromkeys(['valid', 'cb_forest', 'gain', 'gain_cb_forest', 'loss', 'loss_cb_forest',
                    'cb_only_main', 'cb_only_main_gain', 'cb_only_main_loss'], 0)
px_area = None
for t in tiles:
    with rasterio.open(t) as src:
        cb, tr = src.read(1), src.transform
        bounds = src.bounds
    px_area = abs(tr.a * tr.e)
    win = from_bounds(*bounds, transform=hansen.transform)
    off = np.array([win.col_off, win.row_off])
    assert np.allclose(off, np.round(off), atol=1e-3), 'CatBoost and Hansen pixels are not aligned'
    win = win.round_offsets().round_lengths()
    tc, loss, gain, dmask = hansen.read(window=win, boundless=True, fill_value=NODATA)[:, :cb.shape[0], :cb.shape[1]]
    valid = (cb != NODATA) & (tc != NODATA) & (dmask == 1)
    c_for = (cb == FOREST_CODE) & valid
    lost = (loss >= 1) & (loss <= LOSS_UNTIL) if LOSS_UNTIL else np.zeros_like(valid)
    for thr in THRESHOLDS:
        h_for = (tc >= thr) & ~lost & valid
        acc[thr]['both'] += int((c_for & h_for).sum())
        acc[thr]['cb_only'] += int((c_for & ~h_for).sum())
        acc[thr]['h_only'] += int((~c_for & h_for).sum())
        acc[thr]['neither'] += int((valid & ~c_for & ~h_for).sum())
        if thr == MAIN_THRESHOLD:
            cb_only_main = c_for & ~h_for
    g, l = (gain == 1) & valid, (lost & valid)
    gl['valid'] += int(valid.sum());            gl['cb_forest'] += int(c_for.sum())
    gl['gain'] += int(g.sum());                 gl['gain_cb_forest'] += int((g & c_for).sum())
    gl['loss'] += int(l.sum());                 gl['loss_cb_forest'] += int((l & c_for).sum())
    gl['cb_only_main'] += int(cb_only_main.sum())
    gl['cb_only_main_gain'] += int((cb_only_main & g).sum())
    gl['cb_only_main_loss'] += int((cb_only_main & l).sum())
    print(f'{os.path.basename(t)} done')

km2 = lambda n: n * px_area / 1e6

# ------------------------------------------------------------------ agreement by threshold
rows = []
for thr in THRESHOLDS:
    a = acc[thr]
    total = sum(a.values())
    cb_f, h_f = a['both'] + a['cb_only'], a['both'] + a['h_only']
    rows.append({'Hansen threshold (%)': thr,
                 'CatBoost forest (km2)': km2(cb_f), 'Hansen forest (km2)': km2(h_f),
                 'Both forest (km2)': km2(a['both']), 'CatBoost-only (km2)': km2(a['cb_only']),
                 'Hansen-only (km2)': km2(a['h_only']), 'Both non-forest (km2)': km2(a['neither']),
                 'Overall agreement (%)': 100 * (a['both'] + a['neither']) / total,
                 'Hansen forest also CatBoost forest (%)': 100 * a['both'] / h_f if h_f else np.nan,
                 'CatBoost forest also Hansen forest (%)': 100 * a['both'] / cb_f if cb_f else np.nan})
thr_tab = pd.DataFrame(rows).round(2)
thr_tab.to_csv(OUT_DIR + 'hansen_threshold_agreement.csv', index=False)
print('\n=== Agreement by Hansen tree-cover threshold (loss 2001-'
      f'{2000 + LOSS_UNTIL} removed) ===\n', thr_tab.to_string(index=False))

# ------------------------------------------------------------------ Hansen gain / loss
gl_tab = pd.DataFrame([
    {'Item': 'Hansen gain 2000-2012', 'Area (km2)': km2(gl['gain']),
     'Mapped as forest by CatBoost (%)': 100 * gl['gain_cb_forest'] / gl['gain'] if gl['gain'] else np.nan},
    {'Item': f'Hansen loss 2001-{2000 + LOSS_UNTIL}', 'Area (km2)': km2(gl['loss']),
     'Mapped as forest by CatBoost (%)': 100 * gl['loss_cb_forest'] / gl['loss'] if gl['loss'] else np.nan},
    {'Item': f'CatBoost-only forest (threshold {MAIN_THRESHOLD}%)', 'Area (km2)': km2(gl['cb_only_main']),
     'Mapped as forest by CatBoost (%)': 100.0},
    {'Item': '  of which Hansen gain', 'Area (km2)': km2(gl['cb_only_main_gain']),
     'Mapped as forest by CatBoost (%)': np.nan},
    {'Item': '  of which Hansen loss', 'Area (km2)': km2(gl['cb_only_main_loss']),
     'Mapped as forest by CatBoost (%)': np.nan},
]).round(2)
gl_tab['Share of CatBoost-only (%)'] = [np.nan, np.nan, 100.0,
                                        round(100 * gl['cb_only_main_gain'] / gl['cb_only_main'], 2),
                                        round(100 * gl['cb_only_main_loss'] / gl['cb_only_main'], 2)]
gl_tab.to_csv(OUT_DIR + 'hansen_gain_loss.csv', index=False)
print('\n=== Hansen gain and loss vs CatBoost forest ===\n', gl_tab.to_string(index=False))

# ------------------------------------------------------------------ figure
fig, ax = plt.subplots(figsize=(5.2, 3.2))
x = np.arange(len(THRESHOLDS))
parts = [('Both forest (km2)', 'Both forest', '#1B7837'),
         ('CatBoost-only (km2)', 'CatBoost only', '#A6DBA0'),
         ('Hansen-only (km2)', 'Hansen only', '#E6C229')]
bottom = np.zeros(len(THRESHOLDS))
for col, lab, color in parts:
    ax.bar(x, thr_tab[col], 0.6, bottom=bottom, label=lab, color=color, edgecolor='white', lw=0.5)
    bottom += thr_tab[col].values
ax.set_xticks(x, [f'{t}%' for t in THRESHOLDS], fontsize=8)
ax.set_xlabel('Hansen tree-cover threshold', fontsize=9)
ax.set_ylabel('Area (km²)', fontsize=9)
ax.tick_params(labelsize=8)
ax.grid(axis='y', alpha=0.3, lw=0.5)
ax.legend(fontsize=8, frameon=False)
fig.tight_layout()
fig.savefig(OUT_DIR + 'Fig_hansen_thresholds.png', dpi=600, bbox_inches='tight')
fig.savefig(OUT_DIR + 'Fig_hansen_thresholds.pdf', bbox_inches='tight')
plt.show()
print('\nFiles saved to:', OUT_DIR)

In [ ]:
# Run the configuration cell at the top first.
import io
import numpy as np, pandas as pd
import matplotlib.pyplot as plt

# Folder on your Google Drive where the figure is saved (same as in the Hansen notebook)
OUT_DIR = RESULTS_DIR + 'hansen_comparison/'
from matplotlib.ticker import FuncFormatter

# In the Colab notebook, replace this block by: thr_tab = pd.read_csv(OUT_DIR + 'hansen_threshold_agreement.csv')
thr_tab = pd.read_csv(io.StringIO("""Hansen threshold (%),CatBoost forest (km2),Hansen forest (km2),Both forest (km2),CatBoost-only (km2),Hansen-only (km2),Both non-forest (km2),Overall agreement (%),Hansen forest also CatBoost forest (%),CatBoost forest also Hansen forest (%)
10,8670.97,1859.84,1633.82,7037.15,226.02,18764.32,73.74,87.85,18.84
20,8670.97,1245.86,1120.43,7550.55,125.43,18864.91,72.25,89.93,12.92
30,8670.97,974.73,885.68,7785.29,89.05,18901.29,71.53,90.86,10.21
50,8670.97,557.61,513.66,8157.31,43.95,18946.39,70.35,92.12,5.92"""))

INK, MUTED, GRID = '#222222', '#555555', '#E3E3E3'
C_BOTH, C_HONLY = '#1B7837', '#E6C229'
plt.rcParams.update({'font.family': 'DejaVu Sans', 'font.size': 8.5, 'axes.edgecolor': '#444444',
                     'axes.linewidth': 0.8, 'xtick.color': INK, 'ytick.color': INK})

thr = thr_tab['Hansen threshold (%)'].values
x = np.arange(len(thr))
both = thr_tab['Both forest (km2)'].values * 100          # km2 -> ha
honly = thr_tab['Hansen-only (km2)'].values * 100
cb_total = thr_tab['CatBoost forest (km2)'].iloc[0] * 100
fmt = lambda v: f'{v:,.0f}'

fig, (a1, a2) = plt.subplots(1, 2, figsize=(7.4, 3.3), gridspec_kw={'width_ratios': [1.05, 1]})

# (a) Hansen forest area, split by CatBoost agreement
w = 0.72
a1.bar(x, both, w, color=C_BOTH, edgecolor='white', lw=1.2, label='Also mapped as forest by CatBoost', zorder=3)
a1.bar(x, honly, w, bottom=both, color=C_HONLY, edgecolor='white', lw=1.2, label='Hansen only', zorder=3)
for i in range(len(x)):
    a1.text(x[i], both[i] / 2, fmt(both[i]), ha='center', va='center', color='white', fontsize=7, zorder=4)
    a1.text(x[i], both[i] + honly[i] + 2500, f'{fmt(honly[i])}', ha='center', va='bottom', color=INK, fontsize=7.5, zorder=4)
a1.set_xticks(x, [f'{t}%' for t in thr])
a1.set_xlabel('Hansen tree-cover threshold')
a1.set_ylabel('Hansen forest area (ha)')
a1.yaxis.set_major_formatter(FuncFormatter(lambda v, _: f'{v:,.0f}'))
a1.set_ylim(0, 215000)
a1.grid(axis='y', color=GRID, lw=0.6, zorder=0); a1.set_axisbelow(True)
for s in ('top', 'right'): a1.spines[s].set_visible(False)
a1.legend(loc='upper right', frameon=False, fontsize=7.5, handlelength=1.2)
a1.set_title('(a) Hansen forest area', loc='left', fontsize=9, fontweight='bold')

# (b) agreement metrics
series = [('Hansen forest also CatBoost forest (%)', 'Hansen forest mapped as forest by CatBoost', C_BOTH, 'o'),
          ('Overall agreement (%)', 'Overall agreement (forest/non-forest)', '#4D4D4D', 's'),
          ('CatBoost forest also Hansen forest (%)', 'CatBoost forest overlapping Hansen forest', '#B8860B', 'D')]
for col, lab, c, m in series:
    y = thr_tab[col].values
    a2.plot(x, y, color=c, lw=2, marker=m, ms=5.5, mec='white', mew=1, label=lab, zorder=3)
    for xi, yi in zip(x, y):
        a2.text(xi, yi + 2.6, f'{yi:.2f}', ha='center', va='bottom', fontsize=7, color=INK)
a2.set_xticks(x, [f'{t}%' for t in thr])
a2.set_xlabel('Hansen tree-cover threshold')
a2.set_ylabel('Percentage (%)')
a2.set_ylim(0, 105); a2.set_xlim(-0.35, len(x) - 0.65)
a2.grid(axis='y', color=GRID, lw=0.6, zorder=0); a2.set_axisbelow(True)
for s in ('top', 'right'): a2.spines[s].set_visible(False)
a2.legend(loc='center left', bbox_to_anchor=(0.0, 0.47), frameon=False, fontsize=7.2, handlelength=1.8)
a2.set_title('(b) Agreement between products', loc='left', fontsize=9, fontweight='bold')

fig.tight_layout(w_pad=2.2)
fig.savefig(OUT_DIR + 'Fig14_hansen_thresholds.png', dpi=600, bbox_inches='tight', facecolor='white')
fig.savefig(OUT_DIR + 'Fig14_hansen_thresholds.pdf', bbox_inches='tight', facecolor='white')
plt.show()
print('Saved to:', OUT_DIR + 'Fig14_hansen_thresholds.png')
print('CatBoost forest (constant):', fmt(cb_total), 'ha')